# Space-Time Accessibility Results with Sensitivity Analysis (Refactored)

**Changes from 7-sp-accessibility-results-organizer.ipynb:**
- Filters by time budget in post-processing (not pre-computed)
- Supports sensitivity analysis:
  - Time budgets: 60, 75, 90, 120 min
  - Departure times: 16:00, 17:00, 18:00
- Outputs STA for each parameter combination

In [1]:
%load_ext autoreload
%autoreload 2
%cd D:\netmob25

D:\netmob25


In [2]:
import os
import pandas as pd
import numpy as np
from pathlib import Path
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

## Configuration

In [5]:
# Directories
DATA_DIR = Path("dbs/sp_accessibility_v2")
OUTPUT_DIR = Path("results/sensitivity")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Sensitivity analysis parameters
TIME_BUDGETS = [60, 75, 90, 120]  # minutes
DEPARTURE_HOURS = [16, 17, 18]
MODES = ['pt', 'car']

# Baseline parameters (for main analysis)
BASELINE_BUDGET = 90
BASELINE_HOUR = 17

print(f"Input directory: {DATA_DIR}")
print(f"Output directory: {OUTPUT_DIR}")
print(f"\nSensitivity parameters:")
print(f"  Time budgets: {TIME_BUDGETS}")
print(f"  Departure hours: {DEPARTURE_HOURS}")
print(f"  Modes: {MODES}")

Input directory: dbs\sp_accessibility_v2
Output directory: results\sensitivity

Sensitivity parameters:
  Time budgets: [60, 75, 90, 120]
  Departure hours: [16, 17, 18]
  Modes: ['pt', 'car']


## 1. Load base data

In [7]:
# Load time budget data
df_budget = pd.read_csv(DATA_DIR / "data" / "time_budget.csv")
print(f"Time budget data: {len(df_budget)} individuals")

# Get list of individual IDs by mode (for proper pct_nonzero calculation)
all_ids = df_budget['ID'].unique()
car_ids = df_budget[df_budget['is_car'] == 1]['ID'].unique()
pt_ids = df_budget[df_budget['is_car'] == 0]['ID'].unique()

print(f"Unique individuals: {len(all_ids)}")
print(f"  Car users: {len(car_ids)}")
print(f"  PT users: {len(pt_ids)}")

Time budget data: 2457 individuals
Unique individuals: 2457
  Car users: 914
  PT users: 1543


## 2. Define STA computation function

In [5]:
def compute_sta(mode, hour, time_budget, df_budget, data_dir):
    """
    Compute Space-Time Accessibility for given parameters.

    STA = count of unique POIs reachable within REMAINING time budget.
    Remaining time = time_budget - time_hw * 2 (after round-trip commute)
    Constraint: time_wk + time_kh <= remaining_time

    Parameters:
    -----------
    mode : str
        'pt' or 'car'
    hour : int
        Departure hour (16, 17, 18)
    time_budget : int
        Total time budget in minutes (60, 75, 90, 120)
    df_budget : DataFrame
        Individual time budgets with time_hw (commute time)
    data_dir : Path
        Directory containing travel time files

    Returns:
    --------
    DataFrame with columns: ID, ak (accessibility count), mode, hour, budget
    """
    key = f"{mode}_{hour:02d}"

    # Load WK travel times
    wk_file = data_dir / f"tt_wk_{mode}_{hour:02d}.csv"
    if not wk_file.exists():
        print(f"  Warning: {wk_file} not found")
        return None

    df_wk = pd.read_csv(wk_file)
    df_wk.rename(columns={
        'from_id': 'ID',
        'to_id': 'poi_id',
        'travel_time_p50': 'time_wk'
    }, inplace=True)

    # Load KH travel times
    kh_file = data_dir / f"tt_kh_{key}.csv"
    if not kh_file.exists():
        print(f"  Warning: {kh_file} not found")
        return None

    df_kh = pd.read_csv(kh_file)
    df_kh.rename(columns={
        'from_id': 'poi_id',
        'to_id': 'ID',
        'travel_time_p50': 'time_kh'
    }, inplace=True)

    # Merge WK and KH
    df_merged = df_wk.merge(df_kh[['ID', 'poi_id', 'time_kh']],
                            on=['ID', 'poi_id'],
                            how='inner')

    # Compute total travel time for trip chain (W->K + K->H)
    df_merged['total_travel'] = df_merged['time_wk'] + df_merged['time_kh']

    # Merge with budget data to get individual commute time
    df_merged = df_merged.merge(
        df_budget[['ID', 'time_hw']],
        on='ID',
        how='left'
    )

    # Compute individual remaining time: budget - round-trip commute
    df_merged['remaining_time'] = time_budget - df_merged['time_hw'] * 2

    # Filter: trip chain must fit within individual's remaining time
    # Also exclude individuals with no valid remaining time (NA or <= 0)
    df_accessible = df_merged[
        (df_merged['remaining_time'] > 0) &
        (df_merged['total_travel'] <= df_merged['remaining_time'])
    ]

    # Count unique POIs per individual
    df_sta = df_accessible.groupby('ID')['poi_id'].nunique().reset_index()
    df_sta.columns = ['ID', 'ak']

    # Add parameters
    df_sta['mode'] = mode
    df_sta['hour'] = hour
    df_sta['budget'] = time_budget

    return df_sta

## 3. Compute STA for all parameter combinations

In [6]:
# Option A: One-at-a-time sensitivity analysis
# - Vary budget at fixed hour=17
# - Vary hour at fixed budget=90

results = []

# Vary budget at fixed hour
print("=== Varying time budget (hour=17) ===")
for budget in TIME_BUDGETS:
    for mode in MODES:
        print(f"  Computing: mode={mode}, hour=17, budget={budget}")
        df_sta = compute_sta(mode, BASELINE_HOUR, budget, df_budget, DATA_DIR)
        if df_sta is not None:
            results.append(df_sta)
            print(f"    -> {len(df_sta)} individuals with STA > 0")

# Vary hour at fixed budget (skip hour=17 as already computed)
print("\n=== Varying departure hour (budget=90) ===")
for hour in DEPARTURE_HOURS:
    if hour == BASELINE_HOUR:
        continue  # Already computed above
    for mode in MODES:
        print(f"  Computing: mode={mode}, hour={hour}, budget=90")
        df_sta = compute_sta(mode, hour, BASELINE_BUDGET, df_budget, DATA_DIR)
        if df_sta is not None:
            results.append(df_sta)
            print(f"    -> {len(df_sta)} individuals with STA > 0")

=== Varying time budget (hour=17) ===
  Computing: mode=pt, hour=17, budget=60
    -> 202 individuals with STA > 0
  Computing: mode=car, hour=17, budget=60
    -> 262 individuals with STA > 0
  Computing: mode=pt, hour=17, budget=75
    -> 342 individuals with STA > 0
  Computing: mode=car, hour=17, budget=75
    -> 362 individuals with STA > 0
  Computing: mode=pt, hour=17, budget=90
    -> 496 individuals with STA > 0
  Computing: mode=car, hour=17, budget=90
    -> 425 individuals with STA > 0
  Computing: mode=pt, hour=17, budget=120
    -> 762 individuals with STA > 0
  Computing: mode=car, hour=17, budget=120
    -> 546 individuals with STA > 0

=== Varying departure hour (budget=90) ===
  Computing: mode=pt, hour=16, budget=90
    -> 492 individuals with STA > 0
  Computing: mode=car, hour=16, budget=90
  Computing: mode=pt, hour=18, budget=90
    -> 496 individuals with STA > 0
  Computing: mode=car, hour=18, budget=90


In [7]:
# Combine all results
if len(results) > 0:
    df_all = pd.concat(results, ignore_index=True)
    print(f"Total results: {len(df_all)} rows")
    print(f"\nParameter combinations:")
    print(df_all.groupby(['mode', 'hour', 'budget']).size())
else:
    print("No results computed. Check if travel time files exist.")

Total results: 4385 rows

Parameter combinations:
mode  hour  budget
car   17    60        262
            75        362
            90        425
            120       546
pt    16    90        492
      17    60        202
            75        342
            90        496
            120       762
      18    90        496
dtype: int64


## 4. Fill missing individuals with STA=0

In [8]:
def fill_missing_with_zero(df, car_ids, pt_ids):
    """
    For each parameter combination, ensure all MODE-SPECIFIC individuals are present.
    Missing individuals get STA=0.
    
    Car mode uses car_ids as denominator, PT mode uses pt_ids.
    """
    filled = []
    
    for (mode, hour, budget), group in df.groupby(['mode', 'hour', 'budget']):
        # Use mode-specific IDs
        mode_ids = car_ids if mode == 'car' else pt_ids
        
        existing_ids = set(group['ID'])
        missing_ids = set(mode_ids) - existing_ids
        
        # Add missing with ak=0
        if len(missing_ids) > 0:
            missing_df = pd.DataFrame({
                'ID': list(missing_ids),
                'ak': 0,
                'mode': mode,
                'hour': hour,
                'budget': budget
            })
            group = pd.concat([group, missing_df], ignore_index=True)
        
        filled.append(group)
    
    return pd.concat(filled, ignore_index=True)

if len(results) > 0:
    df_complete = fill_missing_with_zero(df_all, car_ids, pt_ids)
    print(f"Complete results: {len(df_complete)} rows")
    
    # Verify counts
    for mode in ['car', 'pt']:
        n_expected = len(car_ids) if mode == 'car' else len(pt_ids)
        n_actual = len(df_complete[(df_complete['mode'] == mode) & (df_complete['budget'] == 90)])
        print(f"  {mode}: {n_actual} individuals (expected {n_expected})")

Complete results: 12914 rows
  car: 914 individuals (expected 914)
  pt: 4629 individuals (expected 1543)


In [16]:
df_complete.loc[df_complete['mode'] == 'pt', 'ID'].nunique()

1543

## 5. Summary statistics

In [9]:
if len(results) > 0:
    # Summary by parameters
    summary = df_complete.groupby(['mode', 'hour', 'budget']).agg({
        'ak': ['mean', 'median', 'std', lambda x: (x > 0).mean() * 100]
    }).round(2)
    summary.columns = ['mean', 'median', 'std', 'pct_nonzero']
    print("=== STA Summary by Parameters ===")
    print(summary)

=== STA Summary by Parameters ===
                      mean  median       std  pct_nonzero
mode hour budget                                         
car  17   60        561.28     0.0   2015.30        28.67
          75       1731.28     0.0   4626.74        39.61
          90       4068.59     0.0   8203.92        46.50
          120     11547.41  2077.0  14743.95        59.74
pt   16   90        979.37     0.0   2948.55        31.89
     17   60         78.05     0.0    542.33        13.09
          75        340.31     0.0   1468.94        22.16
          90       1024.76     0.0   3042.58        32.15
          120      4156.65     0.0   7080.03        49.38
     18   90       1028.61     0.0   3049.78        32.15


In [17]:
if len(results) > 0:
    # Compare baseline vs alternatives
    print("\n=== Sensitivity Analysis ===")
    
    # Budget sensitivity (at hour=17)
    print("\nTime Budget Sensitivity (hour=17):")
    budget_sens = df_complete[df_complete['hour'] == BASELINE_HOUR].groupby(['mode', 'budget'])['ak'].mean().unstack()
    print(budget_sens.round(1))
    
    # Hour sensitivity (at budget=90)
    print("\nDeparture Hour Sensitivity (budget=90):")
    hour_sens = df_complete[df_complete['budget'] == BASELINE_BUDGET].groupby(['mode', 'hour'])['ak'].mean().unstack()
    print(hour_sens.round(1))


=== Sensitivity Analysis ===

Time Budget Sensitivity (hour=17):
budget    60      75      90       120
mode                                  
car     561.3  1731.3  4068.6  11547.4
pt       78.1   340.3  1024.8   4156.7

Departure Hour Sensitivity (budget=90):
hour     16      17      18
mode                       
car     NaN  4068.6     NaN
pt    979.4  1024.8  1028.6


## 6. Save results

In [18]:
if len(results) > 0:
    # Save full results
    output_file = OUTPUT_DIR / "sta_sensitivity_full.csv"
    df_complete.to_csv(output_file, index=False)
    print(f"Saved full results to {output_file}")
    
    # Save summary
    summary_file = OUTPUT_DIR / "sta_sensitivity_summary.csv"
    summary.to_csv(summary_file)
    print(f"Saved summary to {summary_file}")
    
    # Save baseline (budget=90, hour=17) for main analysis
    df_baseline = df_complete[
        (df_complete['budget'] == BASELINE_BUDGET) & 
        (df_complete['hour'] == BASELINE_HOUR)
    ][['ID', 'ak', 'mode']]
    baseline_file = OUTPUT_DIR / "sta_baseline.csv"
    df_baseline.to_csv(baseline_file, index=False)
    print(f"Saved baseline results to {baseline_file}")

Saved full results to results\sensitivity\sta_sensitivity_full.csv
Saved summary to results\sensitivity\sta_sensitivity_summary.csv
Saved baseline results to results\sensitivity\sta_baseline.csv


## 7. Create STA transformations for SEM

In [19]:
if len(results) > 0:
    # For baseline parameters, create transformed STA variables
    df_baseline = df_complete[
        (df_complete['budget'] == BASELINE_BUDGET) & 
        (df_complete['hour'] == BASELINE_HOUR)
    ].copy()
    
    # Create unified STA column (each individual has only their mode's STA)
    # Car users get ak_car, PT users get ak_pt
    df_sta_unified = df_baseline[['ID', 'ak', 'mode']].copy()
    df_sta_unified['ak_mode'] = df_sta_unified['ak']  # STA for their actual mode
    
    # Add transformations
    df_sta_unified['ak_log'] = np.log(df_sta_unified['ak'].replace(0, np.nan))
    df_sta_unified['ak_ihs'] = np.arcsinh(df_sta_unified['ak'])
    df_sta_unified['ak_log1p'] = np.log1p(df_sta_unified['ak'])
    
    print("STA transformations (mode-specific):")
    print(df_sta_unified.groupby('mode')[['ak', 'ak_ihs', 'ak_log1p']].describe().round(2))
    
    # Save
    transforms_file = OUTPUT_DIR / "sta_transforms.csv"
    df_sta_unified.to_csv(transforms_file, index=False)
    print(f"\nSaved transformations to {transforms_file}")

STA transformations (mode-specific):
          ak                                                    ak_ihs        \
       count     mean      std  min  25%  50%     75%      max   count  mean   
mode                                                                           
car    914.0  4068.59  8203.92  0.0  0.0  0.0  3223.5  38634.0   914.0  4.07   
pt    1543.0  1024.76  3042.58  0.0  0.0  0.0   104.5  19882.0  1543.0  2.30   

      ...              ak_log1p                                          
      ...   75%    max    count  mean   std  min  25%  50%   75%    max  
mode  ...                                                                
car   ...  8.77  11.26    914.0  3.75  4.20  0.0  0.0  0.0  8.08  10.56  
pt    ...  5.34  10.59   1543.0  2.08  3.28  0.0  0.0  0.0  4.66   9.90  

[2 rows x 24 columns]

Saved transformations to results\sensitivity\sta_transforms.csv


## Summary

In [ ]:
print("=" * 50)
print("STA computation complete!")
print("=" * 50)
print(f"\nOutput directory: {OUTPUT_DIR}")
print(f"\nFiles created:")
print(f"  - sta_sensitivity_full.csv: All parameter combinations")
print(f"  - sta_sensitivity_summary.csv: Summary statistics")
print(f"  - sta_baseline.csv: Baseline (budget={BASELINE_BUDGET}, hour={BASELINE_HOUR})")
print(f"  - sta_transforms.csv: STA with log/IHS transformations")
print(f"\nSensitivity analysis parameters:")
print(f"  - Time budgets tested: {TIME_BUDGETS}")
print(f"  - Departure hours tested: {DEPARTURE_HOURS}")
print(f"\nNote: Section 8 generates dbs/data_p/commuter_sp_set_r.parquet for notebook 10")

## 8. Generate raw trip chain data for downstream analysis

This section generates `commuter_sp_set_r.parquet` containing the raw trip-level accessibility data needed by notebook 10 (SPA set vs visits analysis).

Columns:
- `ID`: Individual identifier
- `poi_id`: POI identifier  
- `mode`: Transport mode (pt/car)
- `time_wk`: Travel time from work to POI
- `time_kh`: Travel time from POI to home
- `time_left`: Remaining time after trip chain (can be negative for infeasible trips)

In [3]:
def generate_trip_chain_data(mode, hour, time_budget, df_budget, data_dir):
    """
    Generate raw trip chain data with time_left for each ID-POI pair.
    
    Returns DataFrame with: ID, poi_id, mode, time_wk, time_kh, time_left
    """
    key = f"{mode}_{hour:02d}"
    
    # Load WK travel times
    wk_file = data_dir / f"tt_wk_{mode}_{hour:02d}.csv"
    if not wk_file.exists():
        print(f"  Warning: {wk_file} not found")
        return None
    
    df_wk = pd.read_csv(wk_file)
    df_wk.rename(columns={
        'from_id': 'ID',
        'to_id': 'poi_id',
        'travel_time_p50': 'time_wk'
    }, inplace=True)
    
    # Load KH travel times
    kh_file = data_dir / f"tt_kh_{key}.csv"
    if not kh_file.exists():
        print(f"  Warning: {kh_file} not found")
        return None
    
    df_kh = pd.read_csv(kh_file)
    df_kh.rename(columns={
        'from_id': 'poi_id',
        'to_id': 'ID',
        'travel_time_p50': 'time_kh'
    }, inplace=True)
    
    # Merge WK and KH
    df_merged = df_wk.merge(df_kh[['ID', 'poi_id', 'time_kh']],
                            on=['ID', 'poi_id'],
                            how='inner')
    
    # Merge with budget data to get individual commute time
    df_merged = df_merged.merge(
        df_budget[['ID', 'time_hw']],
        on='ID',
        how='left'
    )
    
    # Compute remaining time: budget - round-trip commute - trip chain travel
    # time_left = budget - 2*time_hw - (time_wk + time_kh)
    df_merged['time_left'] = (time_budget 
                               - df_merged['time_hw'] * 2 
                               - df_merged['time_wk'] 
                               - df_merged['time_kh'])
    
    # Add mode
    df_merged['mode'] = mode
    
    # Select and return relevant columns
    return df_merged[['ID', 'poi_id', 'mode', 'time_wk', 'time_kh', 'time_left']]

In [8]:
# Generate trip chain data for baseline parameters (hour=17, budget=90)
print("Generating raw trip chain data for baseline parameters...")
print(f"  Departure hour: {BASELINE_HOUR}")
print(f"  Time budget: {BASELINE_BUDGET} min")

sp_set_list = []
for mode in MODES:
    print(f"  Processing mode: {mode}")
    df_trips = generate_trip_chain_data(mode, BASELINE_HOUR, BASELINE_BUDGET, df_budget, DATA_DIR)
    if df_trips is not None:
        sp_set_list.append(df_trips)
        print(f"    -> {len(df_trips)} trip chains, {df_trips['ID'].nunique()} individuals")

if len(sp_set_list) > 0:
    df_sp_set = pd.concat(sp_set_list, ignore_index=True)
    print(f"\nCombined: {len(df_sp_set)} trip chains")
    print(f"Unique individuals: {df_sp_set['ID'].nunique()}")
    print(f"Unique POIs: {df_sp_set['poi_id'].nunique()}")
    
    # Summary of time_left
    print(f"\ntime_left statistics:")
    print(df_sp_set['time_left'].describe())
    
    # Save to parquet
    output_path = Path("dbs/data_p/commuter_sp_set_r.parquet")
    df_sp_set.to_parquet(output_path, index=False)
    print(f"\nSaved: {output_path}")
else:
    print("No trip chain data generated. Check if travel time files exist.")

Generating raw trip chain data for baseline parameters...
  Departure hour: 17
  Time budget: 90 min
  Processing mode: pt
    -> 23698680 trip chains, 1034 individuals
  Processing mode: car
    -> 25164864 trip chains, 662 individuals

Combined: 48863544 trip chains
Unique individuals: 1696
Unique POIs: 43575

time_left statistics:
count    4.886354e+07
mean    -4.591282e+01
std      3.631126e+01
min     -1.770000e+02
25%     -7.000000e+01
50%     -4.600000e+01
75%     -2.000000e+01
max      8.600000e+01
Name: time_left, dtype: float64

Saved: dbs\data_p\commuter_sp_set_r.parquet
